# Exercise 21 - Splitting the Dataset

Estimated time: **35-40 minutes**

The goal of this exercise is to make use of training, validation, and test datasets when evaluating the effectiveness of a deep learning model on the notMNIST dataset.

The **notMNIST** dataset contains 28x28 pixel images of the letters A-J in various exotic typefaces. notMNIST was created by Yaroslav Bulatov. You can read about notMNIST [here](http://yaroslavvb.blogspot.co.uk/2011/09/notmnist-dataset.html) or download it from [here](http://yaroslavvb.com/upload/notMNIST/) or from [Kaggle](https://www.kaggle.com/lubaroli/notmnist).

notMNIST was inspired by the **MNIST** dataset created by LeCun, Cortes, and Burges. You can read about MNIST [here](http://yann.lecun.com/exdb/mnist/).

First run the code below to read in the notMNIST dataset and plot a few samples.

Use the **T4 GPU** for this exercise

In [ ]:
!wget https://pdl-doulos.s3.us-west-2.amazonaws.com/notMNIST.pickle

In [ ]:
%matplotlib inline
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical

with open('notMNIST.pickle', 'rb') as file:
    data = pickle.load(file)
    train_dataset = data['train_dataset']
    train_labels = data['train_labels']
    valid_dataset = data['valid_dataset']
    valid_labels = data['valid_labels']
    test_dataset  = data['test_dataset']
    test_labels  = data['test_labels']

n_labels = 10

train_dataset = train_dataset.reshape(-1, 784).astype(np.float32)
valid_dataset = valid_dataset.reshape(-1, 784).astype(np.float32)
test_dataset  = test_dataset.reshape(-1, 784).astype(np.float32)
train_labels = to_categorical(train_labels)
valid_labels = to_categorical(valid_labels)
test_labels  = to_categorical(test_labels)

print('Training/validation/test dataset shapes:', train_dataset.shape, valid_dataset.shape, test_dataset.shape)

n_train = train_dataset.shape[0]
n_valid = valid_dataset.shape[0]
n_test = test_dataset.shape[0]

# Plot a few examples so we can visualize the dataset
image_size = 28
n = 15
train_x, train_y = (train_dataset[:n], train_labels[:n])
train_x = train_x.reshape(-1,image_size,image_size)

fig = plt.figure(1, figsize=(15,1))
for i in range(n):
    a = fig.add_subplot(1,n,i+1)
    plt.imshow(train_x[i])
plt.show()

Define a function to build and run a TensorFlow graph with multiple hidden layers and L2 regularization.

In [ ]:
import tensorflow
from tensorflow.keras.models        import Sequential
from tensorflow.keras.layers        import Dense, Input
from tensorflow.keras.regularizers  import l2
from tensorflow.keras.optimizers    import SGD

num_hidden     = 128   # Number of units in each hidden layer
minibatch_size = 100   # Size of the minibatch used for stochastic gradient descent

def build_and_run_graph(num_layers, Lambda, n_epochs, learning_rate):
    tensorflow.keras.backend.clear_session()

    model = Sequential()
    model.add(Input(shape=(image_size*image_size,)))
    model.add(Dense(units=num_hidden, activation='relu', kernel_regularizer=l2(Lambda)))
    for i in range(1,num_layers):
        model.add(Dense(units=num_hidden, activation='relu', kernel_regularizer=l2(Lambda)))
    model.add(Dense(units=n_labels, activation='softmax', kernel_regularizer=l2(Lambda)))
    #model.summary()

    model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=learning_rate), metrics=['accuracy'])

    model.fit(train_dataset, train_labels, epochs=n_epochs, batch_size=minibatch_size, shuffle=True)

Run the above function as follows to establish a baseline for experimenting with hyperparameter values.

In [ ]:
build_and_run_graph(num_layers = 1, Lambda = 0.0, n_epochs = 5, learning_rate = 0.01)

Now modify the function so that it evaluates the graph on the validation dataset once per epoch (the training run is divided into 10 epochs) and evaluates the graph on the test dataset once at the end, printing out the average cost and the the average accuracy for each evaluation. You will need to evaluate the validation and test datasets in minibatches because they are each too large to evaluate in one pass. Experiment with the hyperparameter values (num_layers, Lambda, n_epochs, learning_rate) as you try to reduce the cost and increase the accuracy.

In [ ]:
#

#### Solution

Here is our answer. Do not run the cell below unless you want to see the answer we provide!

<details>
    <summary> Click here to view our answer</summary>

    num_hidden     = 128   # Number of units in each hidden layer
    minibatch_size = 100   # Size of the minibatch used for stochastic gradient descent

    def build_and_run_graph(num_layers, Lambda, n_epochs, learning_rate):
        tensorflow.keras.backend.clear_session()

        model = Sequential()
        model.add(Input(shape=(image_size*image_size,)))
        model.add(Dense(units=num_hidden, activation='relu', kernel_regularizer=l2(Lambda)))
        for i in range(1,num_layers):
            model.add(Dense(units=num_hidden, activation='relu', kernel_regularizer=l2(Lambda)))
        model.add(Dense(units=n_labels, activation='softmax', kernel_regularizer=l2(Lambda)))
        #model.summary()

        model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=learning_rate), metrics=['accuracy'])

        model.fit(train_dataset, train_labels, epochs=n_epochs, batch_size=minibatch_size, shuffle=True,
              validation_data=(valid_dataset, valid_labels))

        loss_and_acc = model.evaluate(test_dataset, test_labels, batch_size=minibatch_size, verbose=0)
        print(f'Test loss = {loss_and_acc[0]:6.3f}, accuracy = {loss_and_acc[1]*100:4.1f}')
    
    build_and_run_graph(num_layers = 1, Lambda = 0.0, n_epochs = 5, learning_rate = 0.01)
    
</details>